# Final PCA: self-referential ancestry filtering + covariate PCs

Replaces the entire 1000G-reference-projection cascade this pipeline used to run
(`submit_pca_r1.ipynb` -> `round1_filter.ipynb` -> `submit_pca_r2.ipynb`/
`round2_filter.ipynb` -> `reverse_pca_aou.ipynb`). That cascade existed to classify
AoU samples against 1000G population labels via Mahalanobis distance to a *reference*
cluster -- necessarily restricted to HM3 variants (for 1000G-comparability), which is
ascertainment-biased toward common EUR variants and much sparser than what's
available. This notebook instead starts from AoU's own premade continental label
(`premade_label_filter.ipynb`) and refines it entirely **within AoU's own data** --
no reference projection, no HM3 restriction, no ID/REF/ALT harmonization against
1000G needed at all.

**Which variants:** the same ACAF-derived, QC'd + LD-pruned panel
`genome_wide_qc_thinning_merge.ipynb` already builds per `BASE_GROUP` for GRM
construction -- one shared build, not a separate variant-selection convention for
PCA. GRM relatedness wants that panel's ~1M-variant density; PCA converges with far
fewer, well-spaced markers, so this notebook further thins it to ~100K (same
calibrate-then-apply `--thin` pattern `king_po_exclusion.ipynb` uses for its own
kinship-specific thinning).

**Two stages, because "which PCA" means two different things here:**
1. **Stage 1 (per `BASE_GROUP`, shared)** -- fits a small PCA (`N_PCS_ELLIPSOID=2`)
   on the *whole* premade-label cohort (`eur` or `afr`), then fits a **self-referential**
   Mahalanobis ellipsoid directly to that cohort's own PC mean/covariance -- no
   reference population needed, since the question is no longer "does this look like
   1000G's CEU/GBR" but "is this sample near the center of AoU's own EUR-premade
   cloud." Different `THRESHOLD_QUANTILE`s per final `SAMPLE_SET` (stringent/default/
   loose) implement the purity sweep entirely here, against the same base cohort --
   `eur`/`eur_stringent`/`eur_loose` all start from and are compared against the
   identical `eur`-premade population, just with different cutoffs. `eur_premade_label`
   skips the ellipsoid step entirely (keeps the whole premade-label cohort, unfiltered)
   -- it exists specifically as the "no PCA-based refinement at all" comparison point.
2. **Stage 2 (per `SAMPLE_SET`, refit within that set's own final members)** -- PCs
   fit on a superset and then subsetted don't correctly reflect a tighter cohort's own
   structure, so once Stage 1 has decided a `SAMPLE_SET`'s final membership, this
   stage reruns `--pca approx` **restricted to exactly those members** (`--keep`,
   cheap -- reuses Stage 1's already-thinned panel, no new pruning), down to
   `N_PCS=10`. These are the PCs that actually get written out as covariates for
   `residualize_phenotypes.ipynb`.

## Compute resource

Same panel `03_grm_shards/grm_shard_timing.ipynb`/`grm_shard_run.ipynb` size for
initially, but both stages here run on the ~100K-variant thinned subset, not the
full ~1M-variant GRM panel -- 8-16 vCPU is plenty, no need to size up to the GRM
notebooks' level.

## Setup

plink2: manual install, same pattern as everywhere else in this repo.

In [ ]:
%%bash
set -e

BIN_DIR="$HOME/bin"
mkdir -p "$BIN_DIR"

if [ ! -x "$BIN_DIR/plink2" ]; then
  # URL is dated; if it 404s, get current link from https://www.cog-genomics.org/plink/2.0/
  PLINK2_URL="https://s3.amazonaws.com/plink2-assets/alpha7/plink2_linux_x86_64_20260504.zip"
  cd /tmp
  wget -q -O plink2.zip "$PLINK2_URL"
  unzip -o -q plink2.zip plink2 -d "$BIN_DIR"
  chmod +x "$BIN_DIR/plink2"
fi

export PATH="$BIN_DIR:$PATH"
plink2 --version
nproc
free -h

In [ ]:
import os

bin_dir = os.path.expanduser("~/bin")
if bin_dir not in os.environ["PATH"].split(":"):
    os.environ["PATH"] = f"{bin_dir}:{os.environ['PATH']}"

N_THREADS = os.cpu_count()

## Sample set configuration

Every final `SAMPLE_SET` maps onto one of 2 `BASE_GROUP`s (`eur`, `afr`) and, for the
EUR family, an `ellipsoid_threshold` -- `None` for `eur_premade_label` means "skip
Stage 1's ellipsoid entirely, keep the whole premade-label cohort." Thresholds are
not chosen from a formal power calculation -- `0.99` (stringent) / `0.999999`
(default) / `0.99999999` (loose) are clearly-tighter/clearly-looser reference points
either side of the default, same spirit as the old round1/round2b thresholds they
replace, just now against AoU's own empirical cloud instead of a 1000G reference.
`afr` keeps a single threshold, consistent with round1/round2's own prior single-AFR-
threshold convention (a pooled, more internally-diverse population needs a generous
cutoff to retain a usable N, same reasoning as before).

In [ ]:
SAMPLE_SETS = {
    "eur":               {"base_group": "eur", "ellipsoid_threshold": 0.999999},
    "eur_stringent":     {"base_group": "eur", "ellipsoid_threshold": 0.99},
    "eur_loose":         {"base_group": "eur", "ellipsoid_threshold": 0.99999999},
    "eur_premade_label": {"base_group": "eur", "ellipsoid_threshold": None},   # no ellipsoid -- whole premade cohort
    "afr":               {"base_group": "afr", "ellipsoid_threshold": 0.999},
}

N_PCS_ELLIPSOID = 2    # Stage 1's Mahalanobis fit dimensionality -- matches every
                        # prior round's own N_PCS convention
N_PCS = 10              # Stage 2's final covariate PC count
PCA_N_SNPS_TARGET = 100_000   # GRM wants ~1M variants; PCA converges with far fewer

CDR_VERSION = "v9"
WORKSPACE_BUCKET = os.path.expanduser(
    "~/workspace/Data from All of Us Controlled Tier /shared-env-pilot"
)
ANCESTRY_BUCKET_DIR = f"{WORKSPACE_BUCKET}/{CDR_VERSION}/01_ancestry_filtering"

LOCAL_WORK_DIR = os.path.expanduser("~/scratch_grm")
os.makedirs(LOCAL_WORK_DIR, exist_ok=True)

## Stage 1: base-group ellipsoid membership

Run once per `BASE_GROUP` (`"eur"`, then `"afr"`) -- produces every EUR-family
`SAMPLE_SET`'s final keep-list in one pass (they all share this same fit), or `afr`'s
single keep-list. Rerunning with a different `BASE_GROUP` does NOT need to precede
every Stage 2 run -- once Stage 1 has been run for a `BASE_GROUP`, every `SAMPLE_SET`
under it is ready for Stage 2.

In [ ]:
BASE_GROUP = "eur"   # <-- change this and rerun for "eur" and "afr"

PANEL_DIR = f"{ANCESTRY_BUCKET_DIR}/genome_wide_panel_{BASE_GROUP}"
MERGED_NAME = f"genome_wide_thinned_{CDR_VERSION}_{BASE_GROUP}"   # pgen form, from genome_wide_qc_thinning_merge.ipynb
MERGED_PREFIX = os.path.join(LOCAL_WORK_DIR, MERGED_NAME)

for ext in ("pgen", "pvar", "psam"):
    bucket_path = f"{PANEL_DIR}/{MERGED_NAME}.{ext}"
    local_path = f"{MERGED_PREFIX}.{ext}"
    assert os.path.isfile(bucket_path), (
        f"missing merged panel: {bucket_path!r} -- run genome_wide_qc_thinning_merge.ipynb's "
        f"merge section for BASE_GROUP={BASE_GROUP!r} first"
    )
    if not os.path.isfile(local_path):
        import shutil
        shutil.copy(bucket_path, local_path)

FINAL_PCA_BUCKET_DIR = f"{PANEL_DIR}/final_pca"
os.makedirs(FINAL_PCA_BUCKET_DIR, exist_ok=True)

PCA_BED_PREFIX = os.path.join(LOCAL_WORK_DIR, f"{MERGED_NAME}_pca_thinned")   # shared by both stages
ELLIPSOID_PCA_PREFIX = os.path.join(LOCAL_WORK_DIR, f"final_pca_ellipsoid_{CDR_VERSION}_{BASE_GROUP}")

print(MERGED_PREFIX)
print(FINAL_PCA_BUCKET_DIR)

### Further thin for PCA

Same calibrate-then-apply `--thin` pattern as `king_po_exclusion.ipynb`'s own
kinship-specific thinning of this same panel (50K there, for a different downstream
need). Shared by both stages -- Stage 2 reuses this exact thinned panel, restricted
by `--keep` to a `SAMPLE_SET`'s own final members, rather than re-thinning per
sample set.

In [ ]:
%%bash -s "$MERGED_PREFIX" "$PCA_BED_PREFIX" "$PCA_N_SNPS_TARGET" "$N_THREADS"
set -e
MERGED_PREFIX=$1
PCA_BED_PREFIX=$2
N_TARGET=$3
THREADS=$4

if [ -s "${PCA_BED_PREFIX}.pgen" ]; then
  echo "already thinned, skipping"
else
  N_CURRENT=$(($(wc -l < "${MERGED_PREFIX}.pvar") - 1))
  THIN_P=$(python3 -c "print(min(1.0, ${N_TARGET} / ${N_CURRENT}))")
  echo "current SNPs: $N_CURRENT, target: $N_TARGET, thin_p: $THIN_P"

  plink2 \
    --pfile "$MERGED_PREFIX" \
    --thin "$THIN_P" \
    --threads "$THREADS" \
    --make-pgen \
    --out "$PCA_BED_PREFIX"
fi

echo "Thinned SNP count:"
awk 'END{print NR-1}' "${PCA_BED_PREFIX}.pvar"

### Fit Stage 1's small PCA + self-referential ellipsoid

`--pca approx`: fits on the whole premade-label cohort directly (no subsampling) --
plink2's exact PCA algorithm doesn't scale to a cohort this size, so this uses the
randomized/Blanczos algorithm (Galinsky et al. 2016, "fastPCA"), same as every other
full-cohort fit in this pipeline. The ellipsoid is fit to THIS cohort's own PC
mean/covariance -- no reference population, no `pop`/`super_pop` labels needed at
all, which is the whole point of the self-referential approach.

In [ ]:
%%bash -s "$PCA_BED_PREFIX" "$ELLIPSOID_PCA_PREFIX" "$N_THREADS" "$N_PCS_ELLIPSOID"
set -e
PCA_BED_PREFIX=$1
ELLIPSOID_PCA_PREFIX=$2
THREADS=$3
NPCS=$4

if [ -s "${ELLIPSOID_PCA_PREFIX}.eigenvec" ]; then
  echo "already fit, skipping"
else
  plink2 \
    --pfile "$PCA_BED_PREFIX" \
    --nonfounders \
    --pca approx "$NPCS" \
    --threads "$THREADS" \
    --out "$ELLIPSOID_PCA_PREFIX"
fi

ls -lh "${ELLIPSOID_PCA_PREFIX}".*

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import chi2

ellipsoid_pcs = pd.read_csv(f"{ELLIPSOID_PCA_PREFIX}.eigenvec", sep=r"\s+")
id_col = "#IID" if "#IID" in ellipsoid_pcs.columns else "IID"
pc_cols_ellipsoid = [f"PC{i}" for i in range(1, N_PCS_ELLIPSOID + 1)]
assert all(c in ellipsoid_pcs.columns for c in pc_cols_ellipsoid), \
    f"missing PC columns, have: {list(ellipsoid_pcs.columns)}"

pcs = ellipsoid_pcs[pc_cols_ellipsoid].values
mean = pcs.mean(axis=0)
cov_inv = np.linalg.inv(np.cov(pcs, rowvar=False))

def mahal(x):
    d = x - mean
    return np.sqrt(d @ cov_inv @ d)

ellipsoid_pcs["mahal"] = [mahal(row) for row in pcs]

for sample_set, cfg in SAMPLE_SETS.items():
    if cfg["base_group"] != BASE_GROUP:
        continue
    threshold_quantile = cfg["ellipsoid_threshold"]

    if threshold_quantile is None:
        # eur_premade_label: no ellipsoid filtering at all -- keep the whole
        # premade-label cohort unfiltered, by construction of what this sample set is
        keep_ids = ellipsoid_pcs[id_col]
        prob_tag = "unfiltered"
    else:
        threshold = np.sqrt(chi2.ppf(threshold_quantile, df=N_PCS_ELLIPSOID))
        keep_ids = ellipsoid_pcs.loc[ellipsoid_pcs["mahal"] <= threshold, id_col]
        prob_tag = f"p{threshold_quantile * 100:g}"

    keep_path = os.path.join(FINAL_PCA_BUCKET_DIR, f"final_keep_ids_{sample_set}_{prob_tag}.txt")
    keep_ids.to_csv(keep_path, index=False, header=False)
    print(f"[{sample_set}] {len(keep_ids)}/{len(ellipsoid_pcs)} retained -> {keep_path}")

## Stage 2: per-sample-set final PCA for covariates

Run once per `SAMPLE_SET` (all 5) -- restricts Stage 1's already-thinned panel to
this sample set's own final keep-list (`--keep`, cheap -- no re-QC/re-pruning), then
refits `--pca approx` down to `N_PCS` within exactly that cohort. This is the fit
that actually gets written out as covariates -- PCs computed on a superset and then
row-subsetted don't correctly reflect a tighter cohort's own structure, so this
deliberately reruns the fit rather than reusing Stage 1's.

In [ ]:
SAMPLE_SET = "eur"   # <-- change this and rerun for each of the 5 sample sets
_cfg = SAMPLE_SETS[SAMPLE_SET]
_base_group = _cfg["base_group"]

_panel_dir = f"{ANCESTRY_BUCKET_DIR}/genome_wide_panel_{_base_group}"
_final_pca_bucket_dir = f"{_panel_dir}/final_pca"
_pca_bed_prefix = os.path.join(LOCAL_WORK_DIR, f"genome_wide_thinned_{CDR_VERSION}_{_base_group}_pca_thinned")
assert os.path.isfile(f"{_pca_bed_prefix}.pgen"), (
    f"missing thinned panel: {_pca_bed_prefix}.pgen -- run Stage 1 for BASE_GROUP={_base_group!r} first"
)

# Stage 1's own keep-list-writing cell derives prob_tag as f"p{threshold*100:g}"
# (or "unfiltered" for eur_premade_label) -- recompute the same way here so this
# cell finds the right file without needing Stage 1's Python variables still in
# memory
_threshold = _cfg["ellipsoid_threshold"]
_prob_tag = "unfiltered" if _threshold is None else f"p{_threshold * 100:g}"
SAMPLE_SET_KEEP_PATH = os.path.join(_final_pca_bucket_dir, f"final_keep_ids_{SAMPLE_SET}_{_prob_tag}.txt")
assert os.path.isfile(SAMPLE_SET_KEEP_PATH), (
    f"missing keep-list: {SAMPLE_SET_KEEP_PATH!r} -- run Stage 1 for BASE_GROUP={_base_group!r} first"
)

SAMPLE_SET_OUT_DIR = os.path.join(_final_pca_bucket_dir, SAMPLE_SET)
os.makedirs(SAMPLE_SET_OUT_DIR, exist_ok=True)
FINAL_PCA_PREFIX = os.path.join(LOCAL_WORK_DIR, f"final_pca_{CDR_VERSION}_{SAMPLE_SET}")

print(SAMPLE_SET_KEEP_PATH)
print(SAMPLE_SET_OUT_DIR)

In [ ]:
%%bash -s "$_pca_bed_prefix" "$SAMPLE_SET_KEEP_PATH" "$FINAL_PCA_PREFIX" "$N_THREADS" "$N_PCS"
set -e
PCA_BED_PREFIX=$1
KEEP_PATH=$2
FINAL_PCA_PREFIX=$3
THREADS=$4
NPCS=$5

plink2 \
  --pfile "$PCA_BED_PREFIX" \
  --keep "$KEEP_PATH" \
  --nonfounders \
  --freq counts \
  --pca approx "$NPCS" \
  --threads "$THREADS" \
  --out "$FINAL_PCA_PREFIX"

ls -lh "${FINAL_PCA_PREFIX}".*

### Scree plot

Quick sanity check -- % variance explained per PC, refit within this exact
`SAMPLE_SET`'s own members.

In [ ]:
import matplotlib.pyplot as plt

eigenval = pd.read_csv(f"{FINAL_PCA_PREFIX}.eigenval", header=None, names=["eigenvalue"])
eigenval["pc"] = range(1, len(eigenval) + 1)
eigenval["pct_variance"] = eigenval["eigenvalue"] / eigenval["eigenvalue"].sum() * 100

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(eigenval["pc"], eigenval["pct_variance"], color="royalblue")
ax.set_xlabel("PC")
ax.set_ylabel("% variance explained")
ax.set_title(f"Final PCA [{SAMPLE_SET}] scree plot")
ax.set_xticks(eigenval["pc"])
plt.tight_layout()
plot_path = os.path.join(SAMPLE_SET_OUT_DIR, f"final_pca_scree_{SAMPLE_SET}.png")
plt.savefig(plot_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved {plot_path}")
print(eigenval[["pc", "eigenvalue", "pct_variance"]].to_string(index=False))

## Write PC covariates for residualize_phenotypes.ipynb

Same `IID PC1 ... PC10` format `residualize_phenotypes.ipynb`'s `PC_PATH` /
`pull_covariates()` expects.

In [ ]:
direct = pd.read_csv(f"{FINAL_PCA_PREFIX}.eigenvec", sep=r"\s+")

_id_col = "#IID" if "#IID" in direct.columns else "IID"
pc_cols = [c for c in direct.columns if c.startswith("PC")]
assert len(pc_cols) == N_PCS, f"expected {N_PCS} PC columns, found {len(pc_cols)}: {pc_cols}"

covariate_table = direct[[_id_col] + pc_cols].rename(columns={_id_col: "IID"})
covariate_table["IID"] = covariate_table["IID"].astype(str)

PC_COVARIATE_PATH = os.path.join(SAMPLE_SET_OUT_DIR, f"final_pca_pc_covariates_{SAMPLE_SET}.txt")
covariate_table.to_csv(PC_COVARIATE_PATH, sep="\t", index=False)
print(f"Wrote {len(covariate_table)} samples' PC1-PC{N_PCS} -> {PC_COVARIATE_PATH}")

## Next steps

Point `residualize_phenotypes.ipynb`'s `KEEP_LIST_PATH` at `SAMPLE_SET_KEEP_PATH`
and `PC_PATH` at `PC_COVARIATE_PATH` above, for whichever `SAMPLE_SET` was just run.
`residualize_phenotypes_round2.ipynb` (the classic-round-2 twin) and round 2b's
`reverse_pca_aou.ipynb` output are both retired -- this notebook is now the only
ancestry-filtering path feeding phenotype residualization.